# Multifidelity testing

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys

from scipy.stats import uniform, multivariate_normal, Normal, norm

sys.path.append(os.path.join(os.getcwd(), "../../"))
sys.path.append(os.path.join(os.getcwd(), "../../smcpy"))
from smcpy.mcmc.vector_mcmc import VectorMCMC
from smcpy.mcmc.vector_mcmc_kernel import VectorMCMCKernel
from smcpy import AdaptiveSampler as Sampler
from smcpy.paths import GeometricPath

from toy_helpers import M_HF, M_LF, generate_noisy_data
from smcpy.mfmc_proposal import MultiFidelityProposal
from smcpy.proposals import MultivarIndependent

In [2]:
# Data generation details
STD_DEV = 0.2
theta_0 = 1/20
theta_1 = 1
THETA_TRUE = np.array([[theta_0, theta_1]])
NUM_PARTICLES = 2_000
np.random.seed(42)
noisy_data = generate_noisy_data(THETA_TRUE, STD_DEV)

## Low fidelity model

Currently we will work with the case where we know the standard deviation of the additive noise term.

In [3]:
priors = [uniform(0.001, 2), uniform(-2, 8)]
vector_mcmc = VectorMCMC(M_LF, noisy_data, priors, STD_DEV)

# initialize from prior
mcmc_kernel = VectorMCMCKernel(vector_mcmc, param_order=("theta_0", "theta_1"))
smc = Sampler(mcmc_kernel=mcmc_kernel, show_progress_bar=True)
reg_step_list, mll_list = smc.sample(
    num_particles=NUM_PARTICLES,
    num_mcmc_samples=5,
    target_ess=0.5
)
reg_phi_list = smc.phi_sequence

[ mutation ratio: 0.982: : 100.00%|██████████| phi: 1.00000/1.0 [00:17<00:00   


In [4]:
lofi_samples = reg_step_list[-1].params

In [5]:
proposal_dist = MultiFidelityProposal(
    lofi_samples, 
    M_LF, 
    noisy_data,
    STD_DEV,
    priors
)

In [6]:
mcmc_kernel = VectorMCMCKernel(
    vector_mcmc, param_order=("a", "b"), path=GeometricPath(proposal=proposal_dist)
)

In [7]:
smc = Sampler(mcmc_kernel=mcmc_kernel, show_progress_bar=True)
prop_step_list, mll_list = smc.sample(
    num_particles=NUM_PARTICLES,
    num_mcmc_samples=5,
    target_ess=0.5,
    # phi_sequence=phi_sequence,
    # ess_threshold=1.0,
)
prop_phi_list = smc.phi_sequence


[ mutation ratio: 0.9835: : 100.00%|██████████| phi: 1.00000/1.0 [00:04<00:00


### Some proposal checks

In [ ]:
dims_here = [d.rvs(1).size for d in priors]

In [ ]:
np.split(lofi_samples, np.cumsum(dims_here)[:-1], axis=1)

In [ ]:
uniform(1,2)

In [ ]:
mf_proposal = MultiFidelityProposal(
    lofi_samples, 
    M_LF, 
    noisy_data,
    STD_DEV
)
og_proposal = MultivarIndependent(uniform(1,2), uniform(1,3))

In [ ]:
len(lofi_samples)

In [ ]:
# testing on original proposal module

# Draw sample
og_test_sample = og_proposal.rvs(num_samples = 1000)

# Eval log pdf
og_log_pdf_res = og_proposal.logpdf(og_test_sample)
print(og_log_pdf_res.shape)


In [ ]:
# testing on mf proposal

# draw sample
mf_test_sample = mf_proposal.rvs(num_samples = 1000, random_state= 42)
# print(len(mf_test_sample))

# Eval log pdf
mf_log_pdf_res = mf_proposal.logpdf(mf_test_sample)
print(mf_log_pdf_res.shape)
